# 10. 공분산과 Gaussian 기하

로봇 센서는 항상 노이즈를 가진다. 노이즈를 단순한 숫자 하나가 아니라 **방향성을 가진 타원**으로 보는 도구가 공분산 행렬이다.

$$\Sigma = E[(x-\mu)(x-\mu)^T]$$

공분산의 고유벡터는 불확실성 타원의 방향, 고유값은 각 방향의 분산 크기다.

**로보틱스 연결:**
- Kalman Filter의 $P$, $Q$, $R$
- SLAM pose graph의 information matrix $\Omega=\Sigma^{-1}$
- Mahalanobis distance 기반 outlier rejection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. 공분산 타원과 고유값 분해

2D Gaussian의 등확률선은 타원이다. 타원의 축은 공분산 행렬의 고유벡터와 일치한다.

In [ ]:
def covariance_ellipse(mu, Sigma, nsig=2.0, n=160):
    vals, vecs = np.linalg.eigh(Sigma)
    vals = np.maximum(vals, 0)
    theta = np.linspace(0, 2*np.pi, n)
    circle = np.vstack([np.cos(theta), np.sin(theta)])
    ellipse = vecs @ np.diag(np.sqrt(vals) * nsig) @ circle + mu[:, None]
    return ellipse, vals, vecs

np.random.seed(1)
mu = np.array([1.0, -0.4])
Sigma = np.array([[0.55, 0.32], [0.32, 0.28]])
samples = np.random.multivariate_normal(mu, Sigma, size=700)
ellipse, vals, vecs = covariance_ellipse(mu, Sigma, nsig=2)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(samples[:,0], samples[:,1], s=10, color='gray', alpha=0.25, label='samples')
ax.plot(ellipse[0], ellipse[1], color='#E85D24', lw=2.8, label='2σ covariance ellipse')
for i, color in enumerate(['#534AB7', '#1D9E75']):
    axis = vecs[:, i] * np.sqrt(vals[i]) * 2
    ax.arrow(mu[0], mu[1], axis[0], axis[1], color=color, head_width=0.05, length_includes_head=True)
    ax.arrow(mu[0], mu[1], -axis[0], -axis[1], color=color, head_width=0.05, length_includes_head=True)
ax.scatter(mu[0], mu[1], color='black', s=70, label='mean')
ax.set_aspect('equal')
ax.grid(alpha=0.25)
ax.set_title('공분산 행렬의 고유벡터 = 불확실성 타원 축')
ax.legend()
plt.savefig('assets/10_covariance_ellipse.png', dpi=150, bbox_inches='tight')
plt.show()

print('eigenvalues=', np.round(vals, 4))
print('eigenvectors=')
print(np.round(vecs, 4))

## 2. 선형변환을 지나면 공분산도 변환된다

로봇 상태가 선형 모델 $y=Ax$를 지나면 평균과 공분산은 다음처럼 변한다.

$$\mu_y=A\mu_x, \qquad \Sigma_y=A\Sigma_xA^T$$

이 식은 Kalman Filter 예측 단계의 핵심이다.

In [ ]:
A = np.array([[1.2, -0.4], [0.5, 0.8]])
mu_y = A @ mu
Sigma_y = A @ Sigma @ A.T
samples_y = samples @ A.T
ellipse_x, _, _ = covariance_ellipse(mu, Sigma, nsig=2)
ellipse_y, _, _ = covariance_ellipse(mu_y, Sigma_y, nsig=2)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].scatter(samples[:,0], samples[:,1], s=8, color='gray', alpha=0.25)
axes[0].plot(ellipse_x[0], ellipse_x[1], color='#E85D24', lw=2.5)
axes[0].scatter(mu[0], mu[1], color='black', s=60)
axes[0].set_title('원래 분포')

axes[1].scatter(samples_y[:,0], samples_y[:,1], s=8, color='gray', alpha=0.25)
axes[1].plot(ellipse_y[0], ellipse_y[1], color='#1D9E75', lw=2.5)
axes[1].scatter(mu_y[0], mu_y[1], color='black', s=60)
axes[1].set_title('선형변환 후: Sigma_y = A Sigma_x A^T')
for ax in axes:
    ax.set_aspect('equal')
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('assets/10_covariance_transform.png', dpi=150, bbox_inches='tight')
plt.show()

print('sample covariance after transform:')
print(np.round(np.cov(samples_y.T), 4))
print('A Sigma A.T:')
print(np.round(Sigma_y, 4))

## 3. Mahalanobis Distance — 공분산을 고려한 거리

Euclidean distance는 모든 방향의 노이즈를 같게 본다. Mahalanobis distance는 불확실성이 큰 방향의 오차를 덜 벌주고, 작은 방향의 오차를 더 크게 본다.

$$d_M(x)=\sqrt{(x-\mu)^T\Sigma^{-1}(x-\mu)}$$

In [ ]:
def mahalanobis(x, mu, Sigma):
    d = x - mu
    return np.sqrt(d @ np.linalg.solve(Sigma, d))

xgrid = np.linspace(-1.6, 3.8, 180)
ygrid = np.linspace(-2.6, 1.8, 180)
X, Y = np.meshgrid(xgrid, ygrid)
D = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        D[i, j] = mahalanobis(np.array([X[i,j], Y[i,j]]), mu, Sigma)

points = np.array([
    [2.0, 0.4],
    [1.2, 0.8],
    [2.2, -1.1],
])

fig, ax = plt.subplots(figsize=(8, 7))
cont = ax.contour(X, Y, D, levels=[1, 2, 3], colors=['#1D9E75', '#534AB7', '#E85D24'], linewidths=2)
ax.clabel(cont, inline=True, fmt='dM=%.0f')
ax.scatter(samples[:,0], samples[:,1], s=7, color='gray', alpha=0.18)
ax.scatter(mu[0], mu[1], color='black', s=70, label='mean')
for p in points:
    ax.scatter(p[0], p[1], s=80)
    ax.text(p[0]+0.04, p[1]+0.04, f'dE={np.linalg.norm(p-mu):.2f}\ndM={mahalanobis(p, mu, Sigma):.2f}', fontsize=9)
ax.set_aspect('equal')
ax.grid(alpha=0.25)
ax.set_title('Mahalanobis distance: 불확실성 방향을 반영한 거리')
ax.legend()
plt.savefig('assets/10_mahalanobis_distance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 정보행렬 — 확실한 방향에 큰 가중치

정보행렬은 공분산의 역행렬이다.

$$\Omega=\Sigma^{-1}$$

SLAM 최적화에서 잔차 $r$의 비용은 보통 $r^T\Omega r$로 쓴다.
공분산이 작은 방향은 정보가 많으므로 더 강하게 벌점이 들어간다.

In [ ]:
Omega = np.linalg.inv(Sigma)
vals_s, vecs_s = np.linalg.eigh(Sigma)
vals_o, vecs_o = np.linalg.eigh(Omega)

print('Sigma=')
print(np.round(Sigma, 4))
print('\nOmega = inv(Sigma)=')
print(np.round(Omega, 4))
print('\nSigma eigenvalues:', np.round(vals_s, 4))
print('Omega eigenvalues:', np.round(vals_o, 4))
print('서로 역수 관계:', np.round(np.sort(vals_s) * np.sort(vals_o)[::-1], 4))

residuals = [np.array([0.4, 0.0]), vecs_s[:,0]*0.4, vecs_s[:,1]*0.4]
for r in residuals:
    print('residual', np.round(r, 3), 'weighted cost=', round(r @ Omega @ r, 4))

## 요약

| 개념 | 수식 | 로보틱스 활용 |
|------|------|---------------|
| 공분산 | $\Sigma=E[(x-\mu)(x-\mu)^T]$ | 상태/센서 불확실성 |
| 고유분해 | $\Sigma=Q\Lambda Q^T$ | 오차 타원 축과 크기 |
| 공분산 전파 | $A\Sigma A^T$ | Kalman 예측, 선형화된 모델 |
| Mahalanobis distance | $\sqrt{r^T\Sigma^{-1}r}$ | outlier rejection |
| 정보행렬 | $\Omega=\Sigma^{-1}$ | SLAM weighted least squares |